# Phase 0 - SAM2 Environment And Sanity Check

This notebook is a secondary Phase 0 helper. Official Phase 0 execution should be command-line reproducible over SSH on RunPod.

Pass criteria:
- Runtime reports the active GPU.
- RTX 3090/4090-class RunPod runs count as target-environment validation; smaller GPUs count as smoke tests only.
- SAM2 imports and loads successfully.
- Single-image inference saves `viz/phase0_inference_check.png`.
- Model memory leaves training headroom.

In [ ]:
!pip install -q 'torch>=2.5.1' 'torchvision>=0.20.1' supervision pycocotools matplotlib pillow
!pip install -q git+https://github.com/facebookresearch/sam2.git

In [ ]:
from pathlib import Path

Path('viz').mkdir(exist_ok=True)
Path('data/phase0').mkdir(parents=True, exist_ok=True)

## Problem 0.1 - Import Check

In [ ]:
import torch
from sam2.build_sam import build_sam2
from sam2.sam2_image_predictor import SAM2ImagePredictor

assert torch.cuda.is_available(), 'CUDA is required for this project phase.'
gpu_name = torch.cuda.get_device_name(0)
print(gpu_name)
print(torch.cuda.memory_allocated() / 1e9)

if any(name in gpu_name for name in ('RTX 3090', 'RTX 4090', '4090', '3090')):
    print('RunPod target-environment check: PASS')
else:
    print(f'WARNING: Project target is RTX 3090/4090-class RunPod GPU, but current GPU is: {gpu_name}')
    print('This run counts as a smoke test only. It verifies install/inference, not final training feasibility.')

## Problem 0.2 - Single Image Inference And Visualization

Place a robot image named `robot_test.jpg` in the current working directory before running this cell.

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

image_path = Path('robot_test.jpg')
assert image_path.exists(), 'Upload a robot image named robot_test.jpg before running inference.'

predictor = SAM2ImagePredictor.from_pretrained('facebook/sam2.1-hiera-large')

image = np.array(Image.open(image_path).convert('RGB'))
h, w = image.shape[:2]

predictor.set_image(image)
masks, scores, _ = predictor.predict(
    point_coords=np.array([[w // 2, h // 2]]),
    point_labels=np.array([1]),
    multimask_output=False,
)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].imshow(image)
axes[0].plot(w // 2, h // 2, 'r*', markersize=12)
axes[0].set_title('Input + prompt point')
axes[0].axis('off')

axes[1].imshow(image)
axes[1].imshow(masks[0], alpha=0.5, cmap='Reds')
axes[1].set_title(f'SAM2 mask score: {scores[0]:.2f}')
axes[1].axis('off')

boundary = masks[0].astype(np.uint8)
contour_overlay = np.zeros_like(image)
contour_overlay[boundary == 1] = [255, 0, 0]

axes[2].imshow(image)
axes[2].imshow(contour_overlay, alpha=0.4)
axes[2].set_title('Mask boundary overlay')
axes[2].axis('off')

plt.tight_layout()
plt.savefig('viz/phase0_inference_check.png', dpi=150, bbox_inches='tight')
plt.show()

coverage = masks[0].sum() / (h * w) * 100
print(f'Mask coverage: {coverage:.1f}% of image')

if coverage <= 0.5 or coverage >= 95:
    raise AssertionError(f'Mask coverage is clearly broken: {coverage:.1f}%')

if not (5 <= coverage <= 40):
    print(f'WARNING: Expected 5-40% mask coverage for a centered medium-size object, got {coverage:.1f}%.')
    print('If the visualization shows a plausible robot mask, this is acceptable for a smoke test.')

## Problem 0.3 - Memory Budget Check

In [ ]:
allocated_gb = torch.cuda.memory_allocated() / 1e9
total_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
remaining_gb = total_gb - allocated_gb

print(f'Model memory: {allocated_gb:.2f} GB')
print(f'Total VRAM:   {total_gb:.2f} GB')
print(f'Remaining:    {remaining_gb:.2f} GB')

if any(name in gpu_name for name in ('RTX 3090', 'RTX 4090', '4090', '3090')):
    assert remaining_gb > 10, f'Need >10 GB VRAM remaining, got {remaining_gb:.2f} GB'
    print('RunPod memory-headroom check: PASS')
else:
    print('Non-target GPU memory check: smoke test only.')
    print('If inference completed and the visualization was saved, this is enough to move local setup forward.')